# 📝 지식 그래프 과제 LV1(기초): 트리플 읽기·파싱·필터·id 해소

> 이 단원의 새 기술을 **하나씩** 확인합니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 트리플 읽기**: 세 칸 꺼내기 · `"(주어, 관계, 목적어)"` 문자열 파싱 · jsonl 로 불러오기
> - **2. 진단하고 거르기**: 관계 하나로 세기 · 관계별 분포 세기 · 근거가 뒷받침하지 않는 것 골라내기 · 허용 목록으로 거르기 · 중복 트리플 합치기
> - **3. 그래프로 옮기기**: 이름을 id 로 바꾸기(못 찾은 것은 후보로) · `MERGE` 세 줄로 매핑
> - **4. 스키마 설계**: 모델에게 줄 서식(`Triple`) · 온톨로지에 관계 한 줄 더하기

## 풀이 방법
1. 맨 위 **준비 셀 세 개**를 먼저 실행하세요.
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: `data/pgx_lv1_triples.jsonl`. 논문 한 편(**PMC13494166**, 암로디핀을 ADHD 치료제로 재활용할 수 있을지 검토한 리뷰)에서 한 추출기가 뽑아 온 트리플 **원출력**입니다. 중복과 규격 밖 관계가 섞여 있어, 읽고 **정제**하는 게 과제입니다.
- **이 논문은 암로디핀을 `AML` 로 줄여 씁니다.** 의학 문헌에서 `AML` 은 보통 급성골수성백혈병을 가리키는 약어라 헷갈리기 쉬운데, 여기서는 **약물(`Compound`)** 입니다. 3-1 에서 이 표기가 사전에 안 붙는 이유가 됩니다.

화이팅!

> **데이터 출처**
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 본문 발췌 (`pgx_*.jsonl`) | PubMed Central Open Access Subset (pmcid 를 각 문서에 적어 두었습니다) | CC BY |
> | 이름 -> id 사전 (`name2id.json`) | Hetionet v1.0 (https://het.io) + RxNav(NLM) 약물 동의어 | CC0 / 공개 |
> | 큐레이션 관계 (`hetionet_curated.jsonl`) | Hetionet v1.0 에서 CC0 출처만 골라낸 부분 | CC0 |
>
> 논문에서 뽑은 관계는 **그 논문이 그렇게 보고했다**는 뜻이고, Hetionet 관계는 **2016년에 정리된** 문헌 근거라는 뜻입니다. 둘 다 "효능이 입증됐다"는 말이 아닙니다. 이 구분을 트리플 속성 `evidence_level` 로 남기는 법은 교안에서 다룹니다.

아래 준비 셀 세 개를 차례로 실행하세요(모델도 Neo4j 도 쓰지 않습니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 온톨로지: 실행만 하세요
# 관계 규격을 모아 두는 한 곳. 관계를 더하거나 고칠 때는 여기만 손댄다
# 각 관계: (주어 타입, 목적어 타입, 판정 기준)
RELATION_SIGNATURES = {
    "TREATS":           ("Compound", "Disease",
                         "약이 질병을 치료한다. "
                         "질병의 원인이나 진행 자체에 작용한다"),
    "PALLIATES":        ("Compound", "Disease",
                         "약이 질병의 증상을 완화한다. "
                         "질병 자체는 그대로 두고 증상만 덜어 준다"),
    "BINDS":            ("Compound", "Gene",
                         "약이 그 유전자의 단백질에 결합한다. "
                         "그 유전자가 이 약의 대사·수송을 맡는다는 진술도 여기에 적는다. "
                         "그 대신 표적·효소·수송체는 가르지 않는다"),
    "UPREGULATES_CG":   ("Compound", "Gene", "약이 그 유전자의 발현을 증가시킨다"),
    "DOWNREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 감소시킨다"),
    "ASSOCIATES":       ("Disease", "Gene", "질병과 유전자 사이에 연관이 보고됐다"),
    "PRESENTS":         ("Disease", "Symptom", "질병이 그 증상으로 나타난다"),
    "INCLUDES":         ("PharmacologicClass", "Compound",
                         "약효 분류가 그 약물을 포함한다. "
                         "그 약이 어느 계열에 속한다는 진술을 여기에 적는다"),
}

# 노드 타입 5종. 지식그래프를 적재한 단원의 레이블과 글자까지 같아야 한다
NODE_TYPES = {"Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"}

In [ ]:
# [제공 코드] 이름 -> id 사전 준비: 이 셀은 실행만 하세요.
# 사전을 어떻게 만드는지는 엔티티 정규화 단원에서 배웁니다. 여기서는 읽어 쓰기만 합니다.
import json
import re
from pathlib import Path

NAME2ID = json.loads(Path("data/name2id.json").read_text(encoding="utf-8"))


# 표기 차이를 눌러 사전 조회용 키로 만든다. lookup_id 함수가 호출한다
def normalize_name(name):
    """이름 매칭용 정규화: 소문자, 앞뒤 공백 제거, 연속 공백 한 칸, 끝의 괄호 주석 제거."""
    s = re.sub(r"\s+", " ", name).strip().lower()
    return re.sub(r"\s*\([^)]*\)$", "", s).strip()


# 이름을 노드 id 로 바꿔 돌려준다. 트리플을 그래프에 적재할 때 쓴다
def lookup_id(name, node_type):
    """이름과 타입으로 Hetionet id 를 찾아 (id, 사유) 두 칸으로 돌려준다.

    사유는 셋 중 하나다. 못 붙은 이유가 둘로 갈리므로 id 만으로는 구별할 수 없다.
      "hit"        붙었다. 첫 칸이 그 id 다
      "miss"       사전에 그 이름이 없다(또는 타입이 다르다). 첫 칸은 None
      "ambiguous"  후보가 둘 이상이라 이름만으로는 못 고른다. 첫 칸은 None
    """
    if node_type == "Gene":
        # 유전자 기호는 대소문자가 곧 뜻이다. 소문자로 누르면 CAT·SET 같은 흔한 단어가 유전자로 잡힌다
        found = NAME2ID["genes"].get(name.strip())
        return (found, "hit") if found else (None, "miss")
    key = normalize_name(name)
    if key in NAME2ID["ambiguous"]:
        return None, "ambiguous"      # 한 이름이 서로 다른 타입 두 곳에 걸린 경우
    entry = NAME2ID["entries"].get(key)
    # 타입까지 맞아야 같은 개체다. obesity 는 Disease 이면서 Symptom 이라 타입을 안 보면 엉뚱하게 붙는다
    if entry and entry["label"] == node_type:
        return entry["id"], "hit"
    return None, "miss"


print("사전 항목:", len(NAME2ID["entries"]),
      "/ 유전자 기호:", len(NAME2ID["genes"]),
      "/ 애매한 이름:", len(NAME2ID["ambiguous"]))


## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 채점에 쓰는 트리플 원출력을 먼저 훑어봅니다.

In [ ]:
# [제공 코드] 트리플 원출력을 먼저 살펴봅니다
raw = Path("data/pgx_lv1_triples.jsonl").read_text(encoding="utf-8").splitlines()
print('행 수:', len(raw))
for line in raw[:3]:
    print(json.loads(line))

---
# 1. 트리플 읽기

트리플이라는 자료 모양을 파이썬에서 다룹니다. 칸을 꺼내고, 문자열을 쪼개고, 파일에서 불러옵니다(교안_01 1절).

## 1-1. 트리플에서 칸 꺼내기
**배경**: 트리플은 `(주어, 관계, 목적어)` 세 칸입니다. 각 칸을 꺼내 쓸 수 있어야 합니다.

**요구사항**:
- 아래 `fact` 튜플에서 주어·관계·목적어를 각각 변수 **`subj`**, **`rel`**, **`obj`** 에 담으세요.

**예시**: `fact = ("AML", "BINDS", "CACNA1C")` 이면 `subj` 는 `"AML"`, `rel` 은 `"BINDS"`, `obj` 는 `"CACNA1C"` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 튜플의 세 원소를 세 변수에 한 번에 나눠 담는다(언패킹).

세부구현:
1. 왼쪽에 변수 세 개를 쉼표로 늘어놓고 오른쪽에 튜플을 둔다.
2. 세 변수를 한 줄로 출력해 확인한다.
```

</details>

In [ ]:
# 문제 1에서 쓸 트리플 하나. 세 칸이 순서대로 주어·관계·목적어다
fact = ("AML", "BINDS", "CACNA1C")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert subj == 'AML' and rel == 'BINDS' and obj == 'CACNA1C', \
    '세 변수의 순서를 확인하세요. 튜플 순서대로 주어·관계·목적어입니다'
print('✅ 통과!')

## 1-2. 트리플 문자열 파싱하기
**배경**: 사람이 적은 `"(주어, 관계, 목적어)"` 문자열을 튜플로 바꿔야 할 때가 있습니다.

**요구사항**:
- 함수 **`parse_triple(text)`** 를 만드세요. `"(주어, 관계, 목적어)"` 형식 문자열을 받아 `(주어, 관계, 목적어)` **튜플**로 돌려줍니다.
- 바깥 괄호를 없애고 쉼표로 나눈 뒤 **양쪽 공백을 제거**하세요.

**예시**: `parse_triple("(ADHD, ASSOCIATES, CACNA1D)")` 는 `("ADHD", "ASSOCIATES", "CACNA1D")` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 괄호를 벗기고 쉼표로 split 한 뒤 각 조각을 strip 한다.

세부구현:
1. strip('()') 로 바깥 괄호를 없앤다.
2. split(',') 로 세 조각으로 나눈다.
3. 각 조각에 strip() 을 적용해 튜플로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert parse_triple('(ADHD, ASSOCIATES, CACNA1D)') == ('ADHD', 'ASSOCIATES', 'CACNA1D'), \
    '괄호를 벗기고 쉼표로 나눴는지, 조각마다 공백을 지웠는지 확인하세요'
assert parse_triple('(AML, BINDS, CACNB1)') == ('AML', 'BINDS', 'CACNB1'), \
    '다른 입력에서도 같은 규칙이 적용돼야 합니다(값을 고정해 두지 마세요)'
print('✅ 통과!')

## 1-3. 추출 결과 개수 세기
**배경**: 파일에서 트리플을 불러와 몇 개가 뽑혔는지 셉니다.

**요구사항**:
- `data/pgx_lv1_triples.jsonl` 을 한 줄씩 읽어 dict 리스트 **`triples`** 로 만들고, 그 개수를 **`n_total`** 에 담으세요.

**예시**: 파일에는 트리플이 **12개** 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 줄 단위로 읽어 각 줄을 json.loads 한다.

세부구현:
1. Path(경로).read_text(encoding='utf-8').splitlines() 로 줄 리스트를 얻는다.
2. 각 줄을 json.loads 해 리스트로 모은다(triples).
3. len(triples) 를 n_total 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_total == 12, '파일의 줄 수와 다릅니다. 빈 줄이 섞이지 않았는지, splitlines 를 썼는지 보세요'
assert isinstance(triples, list) and isinstance(triples[0], dict), \
    '각 줄을 json.loads 해서 dict 로 만들어야 합니다(문자열 그대로 담으면 안 됩니다)'
assert n_total == len(triples), 'n_total 은 triples 의 길이입니다(수를 손으로 적지 마세요)'
assert {'subject', 'relation', 'object', 'evidence'} <= set(triples[0]), \
    '파일에서 읽은 트리플이어야 합니다(칸이 그대로 들어 있어야 합니다)'
print('✅ 통과!')

---
# 2. 진단하고 거르기

원출력에는 규격 밖 관계와 중복이 섞여 있고, 근거가 어긋난 것도 있습니다. **먼저 무엇이 들어 있는지 세어 보고(2-1·2-2), 무엇이 어긋났는지 찾은 뒤(2-3), 그다음 걸러 냅니다(2-4·2-5).** 순서가 반대면 무엇이 왜 빠졌는지 볼 기회가 없어집니다(허용 목록은 교안_01 2-2, 거르는 자리는 교안_02 3-1).

## 2-1. 특정 관계만 세기
**배경**: "이 약이 어느 유전자에 붙는가"만 보고 싶을 때, 관계가 `BINDS` 인 트리플만 셉니다.

**요구사항**:
- `triples` 에서 `relation` 이 **`'BINDS'`** 인 트리플만 골라 리스트 **`binds`** 에 담고, 그 개수를 **`n_binds`** 에 담으세요.

**예시**: `BINDS` 트리플은 **5개** 입니다(중복 하나가 아직 그대로 들어 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 리스트 컴프리헨션으로 relation 이 BINDS 인 것만 남긴다.

세부구현:
1. 리스트 컴프리헨션으로 relation 이 'BINDS' 인 트리플만 남겨 binds 를 만든다.
2. len(binds) 를 n_binds 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_binds == 5, '관계 이름을 정확히 대문자로 비교했는지 확인하세요'
assert all(t['relation'] == 'BINDS' for t in binds), \
    'binds 에 다른 관계가 섞였습니다. 조건을 relation 에만 걸었는지 보세요'
assert n_binds == len(binds), 'n_binds 는 binds 의 길이입니다(수를 손으로 적지 마세요)'
assert binds == [t for t in triples if t['relation'] == 'BINDS'], \
    'binds 는 triples 를 순서 그대로 거른 결과여야 합니다(중복도 그대로 둡니다)'
print('✅ 통과!')

## 2-2. 관계별 개수 집계하기
**배경**: 어떤 관계가 몇 번 나왔는지 세면 추출 분포를 한눈에 봅니다.

**요구사항**:
- `triples` 를 돌며 `relation` 별 개수를 세어 사전 **`by_rel`** 에 담으세요(키=관계 이름, 값=개수).

**예시**: `by_rel['BINDS']` 는 **5**, `by_rel['TREATS']` 는 **1** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 값별 빈도를 세는 표준 도구(collections.Counter)를 쓴다. 사전에 직접 누적해도 된다.

세부구현:
1. collections 에서 Counter 를 가져온다.
2. 각 트리플의 relation 값만 흘려 넣어 빈도를 센다.
3. 그 결과를 dict 로 바꿔 by_rel 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert by_rel['BINDS'] == 5, '관계 이름을 키로, 개수를 값으로 담았는지 확인하세요'
assert by_rel['TREATS'] == 1, '한 번만 나온 관계도 키로 들어 있어야 합니다'
assert by_rel['ASSOCIATES'] == 4, '정제 전 원출력을 세야 합니다(중복을 미리 지우면 수가 달라집니다)'
assert 'MODULATES' in by_rel, '규격 밖 관계도 여기서는 그대로 세어집니다'
assert sum(by_rel.values()) == len(triples), '모든 관계를 빠짐없이 세었는지 확인하세요'
# 원본에서 다시 세어 대조한다(수를 손으로 적으면 여기서 걸린다)
from collections import Counter

assert by_rel == dict(Counter(t['relation'] for t in triples)), \
    'by_rel 은 triples 의 relation 을 빠짐없이 센 결과여야 합니다'
print('✅ 통과!')

## 2-3. 근거가 뒷받침하지 않는 트리플 골라내기
**배경**: 모델은 관계 이름만 지어내는 게 아니라 **근거 문장을 엉뚱하게 붙이기도** 합니다. 근거가 그 트리플을 실제로 뒷받침하는지 확인합니다(교안_02 3-1 후처리).

**요구사항**:
- 트리플 하나를 받아 근거가 뒷받침하는지 판정하는 함수 **`is_grounded(t)`** 를 만드세요. `subject` 와 `object` 가 **둘 다** 그 트리플의 `evidence` 문자열 안에 있으면 `True`, 아니면 `False` 를 돌려줍니다.
- `triples` 중 `is_grounded` 가 `False` 인 것만 리스트 **`ungrounded`** 에 담으세요(원소는 트리플 dict 그대로).

**예시**: `ungrounded` 의 길이는 **1** 이고, 그 트리플의 `relation` 은 `'MODULATES'` 입니다. 근거 문장에 주어가 안 나오기 때문입니다.

> **원출력 `triples` 에 겁니다.** 정제(2-4·2-5)는 이 진단 뒤에 옵니다. 먼저 걸러 버리면 무엇이 왜 빠졌는지 볼 기회가 없어집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 안에 다른 문자열이 있는지는 in 으로 본다. 두 조건을 and 로 묶는다.

세부구현:
1. is_grounded 는 t['subject'] 와 t['object'] 가 t['evidence'] 안에 있는지 둘 다 확인해 반환한다.
2. triples 를 돌며 is_grounded 가 False 인 것만 모아 ungrounded 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert callable(is_grounded), 'is_grounded 를 함수로 만드세요'
assert len(ungrounded) == 1, \
    f'근거가 뒷받침하지 않는 트리플은 1건입니다(지금 {len(ungrounded)}건). 정제 전 triples 에 걸었는지 보세요'
assert ungrounded[0]['relation'] == 'MODULATES', '주어가 근거 문장에 없는 그 트리플이어야 합니다'
# 판정이 정말 도는지 확인한다. 근거가 멀쩡한 것과 목적어만 빠진 것을 하나씩 넣어 본다
assert is_grounded({'subject': 'a', 'object': 'b', 'evidence': 'a and b'}) is True, \
    '주어와 목적어가 둘 다 근거에 있으면 True 여야 합니다'
assert is_grounded({'subject': 'a', 'object': 'b', 'evidence': 'only a here'}) is False, \
    '한쪽이라도 근거에 없으면 False 여야 합니다'
# ungrounded 를 is_grounded 로 골랐는지 본다(관계 이름으로 집어내면 여기서 걸린다)
assert ungrounded == [t for t in triples if not is_grounded(t)], \
    'ungrounded 는 is_grounded 가 False 인 트리플만 순서 그대로 모은 결과여야 합니다'
print('✅ 통과!')

## 2-4. 허용 관계만 남기기
**배경**: 추출 원출력에는 온톨로지에 없는 관계가 섞일 수 있습니다. **허용 관계**에 있는 트리플만 남겨 그래프를 깨끗하게 유지합니다.

**요구사항**:
- `triples` 에서 `relation` 이 **`RELATION_SIGNATURES`**(제공된 온톨로지)에 있는 트리플만 골라 리스트 **`allowed`** 에 담고, 개수를 **`n_allowed`** 에 담으세요.

**예시**: 허용 관계 트리플은 **10개** 입니다(규격 밖 관계 두 개가 빠집니다).

<details><summary>힌트</summary>

```text
접근방법:
- relation 이 시그니처 dict 의 키에 있는지로 거른다(in 연산).

세부구현:
1. 리스트 컴프리헨션으로 relation 이 RELATION_SIGNATURES 에 있는 트리플만 남겨 allowed 를 만든다.
2. len(allowed) 를 n_allowed 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_allowed == 10, '규격 밖 관계 두 개(MODULATES·CAUSES)가 빠졌는지 확인하세요'
# 개수만 보지 않고 원본에서 다시 걸러 대조한다(수를 손으로 적으면 여기서 걸린다)
assert allowed == [t for t in triples if t['relation'] in RELATION_SIGNATURES], \
    'allowed 는 triples 를 순서 그대로 거른 결과여야 합니다'
print('✅ 통과!')

## 2-5. 중복 트리플 제거하기
**배경**: 같은 사실이 두 번 뽑히기도 합니다. `(주어, 관계, 목적어)` 가 같으면 같은 트리플입니다.

**요구사항**:
- `triples` 를 `(subject, relation, object)` 튜플의 <strong>집합(set)</strong>으로 만들어 **`unique`** 에 담고, 그 개수를 **`n_unique`** 에 담으세요.

**예시**: 서로 다른 트리플은 **11개** 입니다(완전히 같은 트리플 하나가 합쳐집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 각 트리플을 (subject, relation, object) 튜플로 바꿔 set 에 넣는다.

세부구현:
1. 각 트리플의 (subject, relation, object) 를 튜플로 묶어 집합(set) 컴프리헨션으로 unique 를 만든다.
2. 그 집합의 len 을 n_unique 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(unique, set) and len(unique) == 11, \
    '세 칸을 튜플로 묶어 집합 unique 에 담았는지 확인하세요(12개 중 하나가 합쳐집니다)'
assert ('AML', 'BINDS', 'CACNA1C') in unique, \
    '집합 원소는 (subject, relation, object) 세 칸 튜플이어야 합니다'
assert n_unique == len(unique), 'n_unique 는 집합 unique 의 길이입니다'
assert unique == {(t['subject'], t['relation'], t['object']) for t in triples}, \
    'unique 는 triples 전체를 세 칸 튜플로 바꾼 집합이어야 합니다'
print('✅ 통과!')

---
# 3. 그래프로 옮기기

정제한 트리플의 이름을 id 로 바꾸고, 그래프에 넣을 `MERGE` 문으로 옮깁니다(교안_01 1절·2-1).

## 3-1. 이름을 id 로 바꾸기
**배경**: 그래프에 넣으려면 이름이 아니라 **id** 가 있어야 합니다. 제공된 `lookup_id(이름, 타입)` 가 사전을 찾아 `(id, 사유)` 두 칸을 돌려줍니다. 못 찾으면 첫 칸이 `None` 입니다.

**요구사항**:
- `allowed` 의 트리플에서 **주어와 목적어에 나온 서로 다른 (이름, 타입) 쌍**을 모아 각각 조회하세요(같은 이름이 여러 트리플에 나와도 한 번만 조회합니다).
- id 를 찾은 것은 사전 **`resolved`** 에 `{이름: id}` 로 담으세요.
- 못 찾은 이름은 리스트 **`candidates`** 에 담고 **정렬**하세요(`sorted`).

**예시**: `resolved['CACNA1C']` 는 `'Gene::775'` 이고, `candidates` 는 `['ADHD', 'AML']` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 주어 쪽 (이름, 타입) 과 목적어 쪽 (이름, 타입) 을 한 집합으로 모은 뒤 하나씩 조회한다.

세부구현:
1. allowed 를 돌며 (subject, subject_type) 과 (object, object_type) 을 집합에 모은다.
2. 그 집합을 돌며 lookup_id 를 호출해 두 칸 중 첫 칸(id)만 받는다.
3. 결과가 있으면 resolved 사전에, 없으면 임시 리스트에 이름을 담는다.
4. 임시 리스트를 sorted 로 정렬해 candidates 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert resolved.get('CACNA1C') == 'Gene::775', \
    '유전자는 대소문자를 그대로 두고 조회해야 합니다(소문자로 누르면 사전에서 못 찾습니다)'
assert len(resolved) == 4, '허용 관계 트리플에 나온 이름 중 사전에 있는 것은 유전자 4개뿐입니다'
assert sorted(set(candidates)) == ['ADHD', 'AML'], \
    '못 찾은 이름만 담았는지 확인하세요(사전에 없는 이름은 ADHD 와 AML 둘입니다)'
assert candidates == ['ADHD', 'AML'], \
    'candidates 에 같은 이름이 여러 번 들어 있지 않은지 보세요. 한 이름이 여러 트리플에 나오면 ' \
    '조회도 여러 번 됩니다(서로 다른 쌍만 모으세요). 마지막에 sorted 로 정렬도 하세요'
# 사전을 정말 조회했는지 원본에서 다시 만들어 대조한다
expected_resolved = {}
for _t in allowed:
    for _name, _type in ((_t['subject'], _t['subject_type']), (_t['object'], _t['object_type'])):
        _found, _ = lookup_id(_name, _type)
        if _found:
            expected_resolved[_name] = _found
assert resolved == expected_resolved, \
    'resolved 는 allowed 의 이름을 lookup_id 로 조회한 결과여야 합니다(값을 손으로 적지 마세요)'
print('✅ 통과!')

## 3-2. 트리플을 Cypher MERGE 문으로
**배경**: 트리플을 그래프에 넣으려면 `MERGE` 문으로 옮겨야 합니다(교안의 `triple_to_cypher`). 노드의 키는 **id** 입니다.

**요구사항**:
- 함수 **`to_merge(subj_id, subj_type, rel, obj_id, obj_type)`** 를 만드세요. 아래 형식의 **문자열**을 돌려줍니다(줄바꿈 포함):

```text
MERGE (a:주어타입 {id: '주어id'})
MERGE (b:목적어타입 {id: '목적어id'})
MERGE (a)-[:관계]->(b)
```

**예시**: `to_merge('Compound::DB00381','Compound','BINDS','Gene::775','Gene')` 의 첫 줄은 `MERGE (a:Compound {id: 'Compound::DB00381'})` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- f-string 세 줄을 줄바꿈(\n)으로 이어 붙인다.

세부구현:
1. 첫 줄: 노드 a 를 주어 타입 레이블과 id 속성으로 MERGE 한다.
2. 둘째 줄: 노드 b 를 목적어 타입 레이블과 id 속성으로 MERGE 한다.
3. 셋째 줄: a 에서 b 로 향하는 관계를 MERGE 한다.
4. Cypher 의 중괄호는 f-string 안에서 두 번씩 쓴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
out = to_merge('Compound::DB00381', 'Compound', 'BINDS', 'Gene::775', 'Gene')
lines = out.splitlines()
assert lines[0] == "MERGE (a:Compound {id: 'Compound::DB00381'})", \
    '첫 줄 형식을 예시와 글자까지 맞추세요(키는 name 이 아니라 id 입니다)'
assert lines[1] == "MERGE (b:Gene {id: 'Gene::775'})", '둘째 줄은 목적어 타입과 목적어 id 입니다'
assert lines[2] == 'MERGE (a)-[:BINDS]->(b)', '관계는 a 에서 b 로 향해야 합니다'
# 다른 입력으로 한 번 더 부른다(기대 출력을 통째로 박아 두면 여기서 걸린다)
other = to_merge('Disease::DOID:1234', 'Disease', 'ASSOCIATES', 'Gene::999', 'Gene')
assert other.splitlines() == ["MERGE (a:Disease {id: 'Disease::DOID:1234'})",
                              "MERGE (b:Gene {id: 'Gene::999'})",
                              'MERGE (a)-[:ASSOCIATES]->(b)'], \
    '다른 입력에서도 같은 규칙이 적용돼야 합니다(값을 고정해 두지 마세요)'
print('✅ 통과!')

---
# 4. 스키마 설계

모델에게 줄 **서식**과 온톨로지에 적을 **시그니처**를 직접 설계합니다(교안_02 1-2 · 교안_01 2-2).

## 4-1. 트리플 구조화 출력 스키마 만들기
**배경**: 모델이 트리플을 **정해진 틀**로 답하게 하려면 Pydantic 모델이 필요합니다. 교안의 `Triple` 과 같은 모양으로 만듭니다. 값이 정해진 칸은 `str` 로 두지 않고 **허용 목록으로 좁힙니다.**

**요구사항**:
- 답안 셀 맨 위에서 `from typing import Literal` 과 `from pydantic import BaseModel, Field` 를 불러오세요(이 노트북의 준비 셀은 둘 다 불러오지 않습니다).
- 허용 목록 두 개를 별칭으로 만드세요. **`RelationName`** 은 `RELATION_SIGNATURES` 의 키로, **`NodeType`** 은 `NODE_TYPES` 를 **정렬한** 값으로 좁힙니다(집합은 순서가 없어 그냥 쓰면 실행마다 순서가 달라집니다).
- `BaseModel` 을 상속한 **`Triple`** 클래스를 만드세요. 칸은 여섯이고 타입은 이렇습니다.
  - **`subject`** `str` · **`subject_type`** `NodeType` · **`relation`** `RelationName` · **`object`** `str` · **`object_type`** `NodeType` · **`evidence`** `str`
- 각 필드에 `Field(description=...)` 를 붙이세요(설명 문구는 자유롭게, 빈 문자열은 안 됩니다).

**예시**: `Triple(subject='AML', subject_type='Compound', relation='BINDS', object='CACNA1C', object_type='Gene', evidence='direct targets of AML')` 이 만들어지고 `.subject` 가 `'AML'` 입니다. 반대로 `relation='MODULATES'` 로 만들면 **`ValidationError`** 가 납니다.

<details><summary>힌트</summary>

```text
접근방법:
- 온톨로지에서 허용 목록을 꺼내 Literal 별칭 둘을 만들고, BaseModel 을 상속해 여섯 칸을 선언한다.

세부구현:
1. 이 셀 맨 위에서 from typing import Literal 과 from pydantic import BaseModel, Field 를 불러온다.
2. RelationName 은 Literal 에 RELATION_SIGNATURES 의 키를 tuple 로 감싸 넘겨 만든다(dict 를 tuple 로 감싸면 키만 나온다).
3. NodeType 도 같은 방식으로 만들되 NODE_TYPES 를 sorted 로 정렬해 순서를 고정한다.
4. class Triple(BaseModel): 아래에 subject·object·evidence 는 str, subject_type·object_type 은 NodeType, relation 은 RelationName 으로 선언한다.
5. 각 필드의 기본값 자리에 Field 를 두고 description 인자에 설명을 적는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
from typing import get_args

from pydantic import ValidationError

for f in ('subject', 'subject_type', 'relation', 'object', 'object_type', 'evidence'):
    assert f in Triple.model_fields, f'{f} 필드가 없습니다. 여섯 칸을 모두 선언했는지 보세요'
    assert Triple.model_fields[f].description, \
        f'{f} 에 Field(description=...) 이 없습니다(설명이 비어 있어도 안 됩니다)'

# 값이 정해진 칸을 str 로 두지 않고 정말 좁혔는지 본다
rel_values = get_args(Triple.model_fields['relation'].annotation)
assert len(rel_values) >= 8 and set(rel_values) <= set(RELATION_SIGNATURES), \
    'relation 은 str 이 아니라 RELATION_SIGNATURES 의 키로 좁힌 Literal 이어야 합니다'
for f in ('subject_type', 'object_type'):
    assert get_args(Triple.model_fields[f].annotation) == tuple(sorted(NODE_TYPES)), \
        f'{f} 는 NODE_TYPES 를 sorted 로 정렬해 좁혀야 합니다(집합을 그대로 쓰면 순서가 흔들립니다)'

sample = Triple(subject='AML', subject_type='Compound', relation='BINDS',
                object='CACNA1C', object_type='Gene', evidence='direct targets of AML')
assert sample.subject == 'AML', '필드 이름을 예시와 똑같이 맞추세요'

# 좁힌 목록이 실제로 막는지 확인한다. 규격 밖 관계는 서식 단계에서 걸려야 한다
try:
    Triple(subject='AML', subject_type='Compound', relation='MODULATES',
           object='CACNA1C', object_type='Gene', evidence='direct targets of AML')
except ValidationError:
    pass
else:
    raise AssertionError('relation 에 MODULATES 를 넣으면 ValidationError 가 나야 합니다')
print('✅ 통과!')

## 4-2. 관계 시그니처 설계하기
**배경**: 우리 그래프에는 관계 타입이 12종 있는데 교안은 8종만 다뤘습니다. 뺀 것 중 **`RESEMBLES_DD`**(증상·유전자가 겹치는 병끼리 이어 둔 관계)를 다시 넣어 봅니다.

**요구사항**:
- **`RELATION_SIGNATURES`** 에 관계 **`RESEMBLES_DD`** 를 추가하세요.
- 시그니처는 `(주어 타입, 목적어 타입, 판정 기준)` 형식입니다. 주어 타입도 목적어 타입도 `"Disease"` 이고, 판정 기준은 자유롭게(한 줄) 적으세요.

**예시**: 추가 후 `RELATION_SIGNATURES['RESEMBLES_DD'][0]` 과 `[1]` 이 둘 다 `"Disease"` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 기존 시그니처와 같은 모양의 3-튜플을 새 관계 이름 키에 넣는다.

세부구현:
1. 시그니처 dict 에 새 키를 하나 만든다(키는 관계 이름).
2. 값으로 (주어 타입, 목적어 타입, 판정 기준) 세 칸 튜플을 넣는다.
3. 넣은 값을 출력해 확인한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
sig = RELATION_SIGNATURES['RESEMBLES_DD']
assert sig[0] == 'Disease' and sig[1] == 'Disease', \
    '(주어 타입, 목적어 타입, 판정 기준) 순서를 확인하세요'
assert isinstance(sig[2], str) and len(sig[2]) > 0, '세 번째 칸에 판정 기준을 한 줄 적으세요'
assert len(RELATION_SIGNATURES) == 9, '기존 8종에 하나를 더한 9종이어야 합니다'
print('✅ 통과!')

---
수고했어요! LV1 에서 트리플 읽기·파싱·필터·중복 제거·id 해소·LPG 매핑·시그니처를 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**해 배치 집계와 온톨로지 주입 추출을 다룹니다.